<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex09.1-burgers-2d/Ex09.1_04_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_09.1 · Notebook 04 — assemble the report

**Paired with L9.1 · Laminar Flow**

Collects every run from notebooks 01–03 into a markdown report with the
configuration table, the error tables, and the questions you must answer.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.1-burgers-2d/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import course_core as cc
import problem as pb
pb.OUTPUT_DIR = cc.keep_outputs("Ex09.1_outputs")


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · Load every run

In [ ]:
import pickle

runs = []
cc.needed('nb01_baseline.pkl', 'nb02_runs.pkl', 'nb03_sweep.pkl')   # on Colab without Drive, asks for the missing files
for f in ("nb01_baseline.pkl", "nb02_runs.pkl", "nb03_sweep.pkl"):
    path = os.path.join(cc.OUTPUT_DIR, f)
    if os.path.exists(path):
        with open(path, "rb") as fh:
            d = pickle.load(fh)
        runs.extend(d if isinstance(d, list) else [d])
    else:
        print(f"  missing: {path}  (run the notebook that writes it)")

print(f"{len(runs)} runs loaded")
for i, r in enumerate(runs, 1):
    print(f"  {i}. {r['config']}  ->  rel L2 (u) {r['mean_u_rel']:.3e}")

## 2 · Write it out

In [ ]:
path = pb.make_report(runs,
                      filename=os.path.join(cc.OUTPUT_DIR, "Ex09.1_report.md"),
                      author="YOUR NAME",
                      notes="Replace this with anything you want recorded.")

# On Colab, download it:
# from google.colab import files; files.download(path)
print(open(path).read()[:1500])

<!-- notebook-questions v1 -->
---

## The questions from notebooks 01 to 03

Every notebook in this set ended with four questions under *Before you move
on*. Copy your answers to them into the cell below — a few sentences each —
and the cell after it adds all 12, each under its question, to the end of the
report you just wrote. On Colab every notebook runs on its own machine, so this
notebook cannot read what you wrote in the others: copying is the only way
across.

Keep the answers short and in your own words. The arrow after each question
names the question on the lecture's Questions slide that it helps answer, so
this section is also your preparation for the oral examination.

In [ ]:
# notebook-questions v1 -- your answers from notebooks 01 to 03 ----------
# Paste each answer between its triple quotes. The question is in the
# comment above it; an empty answer is reported as not answered.

NOTEBOOK_ANSWERS = {

    # ---- notebook 01 · write the residual and the loss ---------------------------
    # 01.1 Where is the error largest — in the smooth regions or along the
    # front? The collocation points here are spread evenly over the whole
    # space-time box and drawn once. Say where they belong on this problem
    # instead, and why a uniform spread spends most of them where nothing
    # happens. (-> L9.1 Q6)
    "01.1": """
""",
    # 01.2 `u + v = 3/2` for the exact solution, everywhere and for all time.
    # Check whether your model obeys it; nothing in the loss asked it to. How
    # could you build the identity into the network, the way a stream function
    # builds in continuity, and what would that guarantee and cost? And for a
    # cylinder with no exact field, what would you check instead? (-> L9.1 Q4,
    # Q8)
    "01.2": """
""",
    # 01.3 The loss has no weight between its terms. Compare the size of the
    # residual and of the boundary mismatch at the end of training and say
    # whether one term is doing all the work. In full Navier–Stokes the
    # continuity residual has no time derivative and is much smaller than the
    # momentum ones: say what continuity constrains, and why an unweighted sum
    # lets it be outvoted. (-> L9.1 Q2)
    "01.3": """
""",
    # 01.4 Burgers is Navier–Stokes with the pressure and the continuity
    # equation removed. Name the term that is left and causes the difficulty,
    # and say how it couples $u$ to $v$ in your two residuals. Then say how the
    # Reynolds number sets the front width $8\nu$, and what that tells you
    # about your chances at higher Re. (-> L9.1 Q1, Q3)
    "01.4": """
""",

    # ---- notebook 02 · the control panel -----------------------------------------
    # 02.1 Run the sampling study, 500 to 20 000 collocation points with
    # everything else fixed. Where does the error plateau? Say why adding more
    # points spread evenly helps less and less on a solution whose difficulty
    # sits in one narrow front, and where you would put the next thousand
    # points instead. (-> L9.1 Q6)
    "02.1": """
""",
    # 02.2 Move the Reynolds slider and watch the viscosity and front width in
    # the readout. Then grow the network without adding points until the
    # readout warns that $N_f/P^*$ is below 10. What does the Reynolds number
    # tell you about your chances, and why does a bigger network on the same
    # points not improve them? (-> L9.1 Q3, Q6)
    "02.2": """
""",
    # 02.3 Compare a shared network for $u$ and $v$ with separate ones at a
    # similar parameter count. Which won, and by how much? Explain the result
    # through the term that couples the two equations, and say why that term is
    # the one that makes flow problems hard. (-> L9.1 Q1)
    "02.3": """
""",
    # 02.4 Vary the L-BFGS epochs from 0 to 800 and note what the handoff buys
    # in error and in wall time. This is a forward problem with known boundary
    # data. When would you use a finite-volume code instead, and what kind of
    # task would still make the PINN the right choice? (-> L9.1 Q10)
    "02.4": """
""",

    # ---- notebook 03 · the Reynolds sweep ----------------------------------------
    # 03.1 Read your error-against-Re plot. At what Re did your model stop
    # following the solution, and how does the front width $8\nu$ compare with
    # the spacing of your collocation points there? Use your answer to say what
    # the Reynolds number tells you about a PINN's chances before you train.
    # (-> L9.1 Q3)
    "03.1": """
""",
    # 03.2 Find a run whose final loss is as low as the low-Re runs but whose
    # error is much worse. Describe what its front profile shows. Why is a
    # smooth, plausible, wrong field more dangerous than a crash, and what
    # would you check on a flow with no exact solution to catch it? (-> L9.1
    # Q8)
    "03.2": """
""",
    # 03.3 The points in this sweep are drawn once, spread evenly, and never
    # moved. For your worst run, say where the points should have gone and why
    # not uniformly. How would residual-adaptive refinement decide that for
    # you? (-> L9.1 Q6)
    "03.3": """
""",
    # 03.4 Given the ceiling you found, when would you hand this problem to a
    # finite-volume code instead of a PINN? Name one task on a flow like this
    # that a finite-volume code cannot do and a PINN can. (-> L9.1 Q10)
    "03.4": """
""",
}


In [ ]:
# notebook-questions v1 -- add the questions and your answers to the report ----------
# Safe to run again: it replaces the section rather than adding a second copy.
import os

NOTEBOOK_QUESTIONS = {
    "01.1": ('01', 'write the residual and the loss', 'Where is the error largest — in the smooth regions or along the front? The collocation points here are spread evenly over the whole space-time box and drawn once. Say where they belong on this problem instead, and why a uniform spread spends most of them where nothing happens.', 'L9.1 Q6'),
    "01.2": ('01', 'write the residual and the loss', '`u + v = 3/2` for the exact solution, everywhere and for all time. Check whether your model obeys it; nothing in the loss asked it to. How could you build the identity into the network, the way a stream function builds in continuity, and what would that guarantee and cost? And for a cylinder with no exact field, what would you check instead?', 'L9.1 Q4, Q8'),
    "01.3": ('01', 'write the residual and the loss', 'The loss has no weight between its terms. Compare the size of the residual and of the boundary mismatch at the end of training and say whether one term is doing all the work. In full Navier–Stokes the continuity residual has no time derivative and is much smaller than the momentum ones: say what continuity constrains, and why an unweighted sum lets it be outvoted.', 'L9.1 Q2'),
    "01.4": ('01', 'write the residual and the loss', 'Burgers is Navier–Stokes with the pressure and the continuity equation removed. Name the term that is left and causes the difficulty, and say how it couples $u$ to $v$ in your two residuals. Then say how the Reynolds number sets the front width $8\\nu$, and what that tells you about your chances at higher Re.', 'L9.1 Q1, Q3'),
    "02.1": ('02', 'the control panel', 'Run the sampling study, 500 to 20 000 collocation points with everything else fixed. Where does the error plateau? Say why adding more points spread evenly helps less and less on a solution whose difficulty sits in one narrow front, and where you would put the next thousand points instead.', 'L9.1 Q6'),
    "02.2": ('02', 'the control panel', 'Move the Reynolds slider and watch the viscosity and front width in the readout. Then grow the network without adding points until the readout warns that $N_f/P^*$ is below 10. What does the Reynolds number tell you about your chances, and why does a bigger network on the same points not improve them?', 'L9.1 Q3, Q6'),
    "02.3": ('02', 'the control panel', 'Compare a shared network for $u$ and $v$ with separate ones at a similar parameter count. Which won, and by how much? Explain the result through the term that couples the two equations, and say why that term is the one that makes flow problems hard.', 'L9.1 Q1'),
    "02.4": ('02', 'the control panel', 'Vary the L-BFGS epochs from 0 to 800 and note what the handoff buys in error and in wall time. This is a forward problem with known boundary data. When would you use a finite-volume code instead, and what kind of task would still make the PINN the right choice?', 'L9.1 Q10'),
    "03.1": ('03', 'the Reynolds sweep', "Read your error-against-Re plot. At what Re did your model stop following the solution, and how does the front width $8\\nu$ compare with the spacing of your collocation points there? Use your answer to say what the Reynolds number tells you about a PINN's chances before you train.", 'L9.1 Q3'),
    "03.2": ('03', 'the Reynolds sweep', 'Find a run whose final loss is as low as the low-Re runs but whose error is much worse. Describe what its front profile shows. Why is a smooth, plausible, wrong field more dangerous than a crash, and what would you check on a flow with no exact solution to catch it?', 'L9.1 Q8'),
    "03.3": ('03', 'the Reynolds sweep', 'The points in this sweep are drawn once, spread evenly, and never moved. For your worst run, say where the points should have gone and why not uniformly. How would residual-adaptive refinement decide that for you?', 'L9.1 Q6'),
    "03.4": ('03', 'the Reynolds sweep', 'Given the ceiling you found, when would you hand this problem to a finite-volume code instead of a PINN? Name one task on a flow like this that a finite-volume code cannot do and a PINN can.', 'L9.1 Q10'),
}

report_md = os.path.join(cc.OUTPUT_DIR, "Ex09.1_report.md")
HEAD = "## Questions from the notebooks"
if not os.path.exists(report_md):
    print("No Ex09.1_report.md yet: run the cell that writes the report first.")
else:
    text = open(report_md, encoding="utf-8").read()
    text = text.split("\n" + HEAD)[0].rstrip() + "\n"
    out = ["", HEAD, "",
           "Each question is tagged with the lecture question it serves.", ""]
    missing, current = [], None
    for key, (nb, title, question, ref) in NOTEBOOK_QUESTIONS.items():
        if nb != current:
            out += [f"### Notebook {nb} · {title}", ""]
            current = nb
        answer = NOTEBOOK_ANSWERS.get(key, "").strip()
        if not answer:
            missing.append(key)
        out += [f"**{key}.** {question} *(→ {ref})*", "",
                answer or "*not answered*", ""]
    with open(report_md, "w", encoding="utf-8") as fh:
        fh.write(text + "\n".join(out))
    print(f"added to {report_md}: {len(NOTEBOOK_QUESTIONS) - len(missing)} of "
          f"{len(NOTEBOOK_QUESTIONS)} answered")
    if missing:
        print("not answered:", ", ".join(missing))


**What you should see.** `added to .../Ex09.1_report.md: 12 of 12 answered`.
Until then it lists the questions still empty, and the report says *not
answered* under each of them — to the marker too. Run this cell again after any
change to the report above it, because rewriting the report removes the
section; then make the PDF.

### The report as a PDF

Moodle shows a PDF inline and a `.md` only as a download, so the cell below
converts the report you just wrote into a PDF (with the figure, if one was
saved) and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex09.1_report.md into Ex09.1_report.pdf, with any figure
# saved as Ex09.1_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex09.1_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md = open(os.path.join(cc.OUTPUT_DIR, "Ex09.1_report.md"), encoding="utf-8").read()
figs = sorted(glob.glob("Ex09.1_report*.png")
              + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex09.1_report*.png")))
if figs:
    imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
    i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
    md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

html = markdown.markdown(md, extensions=["fenced_code", "tables"])
css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                       f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
try:
    from google.colab import files
    files.download(pdf_path)
except ImportError:
    pass


## 3 · Submitting

The generated file contains your configuration table and error tables, plus
five questions under **Your interpretation**. Answer each in a short paragraph
and submit the completed markdown together with the figures you consider
relevant.

The numbers are produced for you. The marks are for the interpretation.

## 4 · Extensions

- Add a variable-viscosity case: make `nu` a function of position and see what breaks.
- Expose `t_end` in the panel and study whether a longer window is harder.
- Replace the exact-solution boundary data with a coarse "measurement" set of
  30 scattered points and check whether the field is still recovered.
- Make `nu` trainable and identify it from the same 30 points. This is the
  inverse problem of slide 18, in miniature.